# Entropy - Pythia Model Family

160M, 410M, 1B, 2.8B, 12B   

## Setup

In [1]:
# Cell 0: Environment Detection
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

Environment: Colab


In [2]:
# Cell 1: Colab Only — Install pinned dependencies
# ⚠️ Restart runtime after running this cell, then skip to Cell 2
if IN_COLAB:
    %pip install -q transformer_lens==2.18.0
    %pip install -q numpy==1.26.4
    %pip install -q transformers==4.57.6

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.9/239.9 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 125.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 138.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4

In [1]:
# Cell 1a: Confirm Transformer Lens version
from importlib.metadata import version
print("TransformerLens version:", version("transformer-lens"))

TransformerLens version: 2.18.0


In [2]:
# Cell 1b: Environment check after session restart
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

Environment: Colab


In [3]:
from pathlib import Path
if IN_COLAB:
    from google.colab import userdata
    token = userdata.get("GH_TMLR")

    repo_owner = "trishasalas"
    repo_name = "tmlr"
    repo_url = f"https://{token}@github.com/{repo_owner}/{repo_name}.git"

    PROJECT_ROOT = Path("/content") / repo_name

    if not PROJECT_ROOT.exists():
        !git clone {repo_url} {PROJECT_ROOT}
else:
    # Local: notebook lives in notebooks/, project root is one level up
    PROJECT_ROOT = Path.cwd().parent
    print(PROJECT_ROOT)

Cloning into '/content/tmlr'...
remote: Enumerating objects: 1907, done.
remote: Counting objects: 100% (319/319), done.
remote: Compressing objects: 100% (247/247), done.
remote: Total 1907 (delta 167), reused 211 (delta 71), pack-reused 1588 (from 1)
Receiving objects: 100% (1907/1907), 10.91 MiB | 29.25 MiB/s, done.
Resolving deltas: 100% (930/930), done.


In [4]:
# Cell 3: Imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Added {PROJECT_ROOT} to sys.path")

import torch
from transformer_lens import HookedTransformer
import transformer_lens.utils as utils

# Device selection
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

Using device: cuda


In [30]:
model_name = "pythia-12b"

In [31]:
# Cell 6: Load Model
model = HookedTransformer.from_pretrained(f"EleutherAI/{model_name}")

print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/9.93G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/9.81G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Loaded pretrained model EleutherAI/pythia-12b into HookedTransformer
Layers: 36
Heads: 40
Hidden size: 5120
Params: 11845.4M


In [32]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Added {PROJECT_ROOT} to sys.path")

### Entropy

In [33]:
from pathlib import Path
import torch
import yaml
import pandas as pd
from src.manifest import write_entropy_manifest

def compute_entropy(logits):
    """H = -Σ P(x_i) log P(x_i)"""
    probs = torch.nn.functional.softmax(logits[0], dim=-1)
    log_probs = torch.log(probs + 1e-10)
    entropy = -torch.sum(probs * log_probs, dim=-1)
    return entropy

prompt_files = [
    'control.yaml',
    'accessibility.yaml',
    'medical.yaml',
    'legal.yaml',
    'finance.yaml',
]

all_results = []
domain_counts = {}

output_dir = PROJECT_ROOT / 'results' / 'entropy' / 'pythia' / model_name
output_dir.mkdir(parents=True, exist_ok=True)

for prompts_file in prompt_files:
    domain = Path(prompts_file).stem
    prompts_path = PROJECT_ROOT / 'data' / prompts_file
    with open(prompts_path, 'r') as f:
        templates = yaml.safe_load(f)
    prompts = templates['prompts']
    print(f"\n--- Running {domain}: {len(prompts)} prompts ---")

    results = []
    for i, case in enumerate(prompts):
        print(f"\r  {i+1}/{len(prompts)}", end="")
        prompt = case['prompt']
        tokens = model.to_tokens(prompt)
        with torch.no_grad():
            logits = model(tokens)
        entropy = compute_entropy(logits)

        results.append({
            'domain': domain,
            'prompt_id': case['prompt_id'],
            'concept': case['concept'],
            'prompt_type': case['prompt_type'],
            'template_type': case['template_type'],
            'prompt': prompt,
            'n_tokens': len(entropy),
            'mean_entropy': round(entropy.mean().item(), 4),
            'last_token_entropy': round(entropy[-1].item(), 4),
            'max_entropy': round(entropy.max().item(), 4),
            'min_entropy': round(entropy.min().item(), 4),
            'model': model_name,
        })
    print()

    domain_df = pd.DataFrame(results)
    filename = f'{model_name}-{domain}.csv'
    domain_df.to_csv(output_dir / filename, index=False)
    all_results.append(domain_df)

    domain_counts[domain] = {
        'expected': len(prompts),
        'written': len(domain_df),
        'file': filename,
    }

print(f"  → {filename}  ({len(domain_df)} rows)")

results_df = pd.concat(all_results, ignore_index=True)

write_entropy_manifest(
    PROJECT_ROOT, output_dir, model_name, model,
    domain_counts, results_df, prompt_files,
)
print(f"\nSaved {len(results_df)} rows across {len(all_results)} domains → {output_dir}")


--- Running control: 44 prompts ---
  44/44

--- Running accessibility: 92 prompts ---
  92/92

--- Running medical: 42 prompts ---
  42/42

--- Running legal: 44 prompts ---
  44/44

--- Running finance: 44 prompts ---
  44/44
  → pythia-12b-finance.csv  (44 rows)

Saved 266 rows across 5 domains → /content/tmlr/results/entropy/pythia/pythia-12b


In [34]:
import os
os.chdir(PROJECT_ROOT)
!git config user.email "trisha@trishasalas.com"
!git config user.name "Trisha Salas"
!git add results/
!git commit -m "entropy results: {model_name}"
!git push

[main 618d108] entropy results: pythia-12b
 6 files changed, 311 insertions(+)
 create mode 100644 results/entropy/pythia/pythia-12b/pythia-12b-accessibility.csv
 create mode 100644 results/entropy/pythia/pythia-12b/pythia-12b-control.csv
 create mode 100644 results/entropy/pythia/pythia-12b/pythia-12b-entropy.md
 create mode 100644 results/entropy/pythia/pythia-12b/pythia-12b-finance.csv
 create mode 100644 results/entropy/pythia/pythia-12b/pythia-12b-legal.csv
 create mode 100644 results/entropy/pythia/pythia-12b/pythia-12b-medical.csv
Enumerating objects: 17, done.
Counting objects: 100% (17/17), done.
Delta compression using up to 12 threads
Compressing objects: 100% (12/12), done.
Writing objects: 100% (12/12), 11.56 KiB | 5.78 MiB/s, done.
Total 12 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/trishasalas/tmlr.git
   c1fb15e..618d108  main -> main


### Delete Model & Clear Cache

In [35]:
# Cell 7: Free memory for next model
import gc
del model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"Memory cleared — GPU: {torch.cuda.memory_allocated()/1e9:.1f}GB allocated")
elif device == "mps":
    torch.mps.empty_cache()
    print("Memory cleared")

Memory cleared — GPU: 0.0GB allocated
